# 🔍 Error Analysis

Where does the model fail? This notebook loads the out-of-fold predictions from
the training set and analyzes residuals by **geohash region**, **hour of day**,
**road type**, and **weather** to find systematic weaknesses.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold
import sys; sys.path.insert(0, '../src')

train = pd.read_csv('../data/train.csv')
y = pd.to_numeric(train['demand'], errors='coerce')
print('train', train.shape)

## Generate out-of-fold predictions

In [ ]:
# quick feature build (inline, minimal)
def mod(t): h,m=str(t).split(':'); return int(h)*60+int(m)
train['mod']=train['timestamp'].map(mod); train['hour']=train['mod']//60
train['gh4']=train['geohash'].astype(str).str.slice(0,4)

feats=['mod','hour','NumberofLanes','Temperature']
for c in ['RoadType','Weather','LargeVehicles','Landmarks','gh4']:
    train[c+'_c']=train[c].astype('category').cat.codes; feats.append(c+'_c')

X=train[feats].fillna(0); folds=list(KFold(5,shuffle=True,random_state=42).split(X))
oof=np.zeros(len(X))
for tr_idx,va_idx in folds:
    m=HistGradientBoostingRegressor(max_iter=600,learning_rate=0.05,max_leaf_nodes=128,random_state=42)
    m.fit(X.iloc[tr_idx],np.log1p(y.iloc[tr_idx]))
    oof[va_idx]=np.clip(np.expm1(m.predict(X.iloc[va_idx])),0,1)

train['pred']=oof; train['residual']=train['demand']-oof; train['abs_err']=train['residual'].abs()
print(f'OOF MAE: {train["abs_err"].mean():.4f}  |  OOF R2: {1-np.var(train["residual"])/np.var(y):.4f}')

## Error by hour of day

In [ ]:
hourly=train.groupby('hour').agg(mae=('abs_err','mean'),bias=('residual','mean'),count=('demand','size')).reset_index()
fig,axes=plt.subplots(1,2,figsize=(14,4)); fig.set_facecolor('#0d1117')
for ax,col,title,color in zip(axes,['mae','bias'],['Mean Absolute Error by Hour','Mean Bias by Hour'],['#f85149','#58a6ff']):
    ax.bar(hourly['hour'],hourly[col],color=color,alpha=0.8)
    ax.set_xlabel('Hour',color='#c9d1d9'); ax.set_title(title,color='#e6edf3',fontweight='bold')
    ax.set_facecolor('#0d1117'); ax.tick_params(colors='#8b949e')
    for s in ax.spines.values(): s.set_color('#30363d')
    if col=='bias': ax.axhline(0,color='#6e7681',ls='--',lw=0.8)
plt.tight_layout(); plt.savefig('../assets/error_by_hour.png',dpi=150,bbox_inches='tight',facecolor='#0d1117'); plt.show()

## Error by road type

In [ ]:
rt=train.groupby('RoadType').agg(mae=('abs_err','mean'),count=('demand','size')).sort_values('mae',ascending=False).reset_index()
fig,ax=plt.subplots(figsize=(10,4)); fig.set_facecolor('#0d1117')
ax.barh(rt['RoadType'],rt['mae'],color='#a371f7',alpha=0.8)
ax.set_xlabel('MAE',color='#c9d1d9'); ax.set_title('Error by Road Type',color='#e6edf3',fontweight='bold')
ax.set_facecolor('#0d1117'); ax.tick_params(colors='#8b949e')
for s in ax.spines.values(): s.set_color('#30363d')
plt.tight_layout(); plt.savefig('../assets/error_by_road.png',dpi=150,bbox_inches='tight',facecolor='#0d1117'); plt.show()

## Error by weather

In [ ]:
wt=train.groupby('Weather').agg(mae=('abs_err','mean'),count=('demand','size')).sort_values('mae',ascending=False).reset_index()
fig,ax=plt.subplots(figsize=(10,4)); fig.set_facecolor('#0d1117')
ax.barh(wt['Weather'].astype(str),wt['mae'],color='#3fb950',alpha=0.8)
ax.set_xlabel('MAE',color='#c9d1d9'); ax.set_title('Error by Weather Condition',color='#e6edf3',fontweight='bold')
ax.set_facecolor('#0d1117'); ax.tick_params(colors='#8b949e')
for s in ax.spines.values(): s.set_color('#30363d')
plt.tight_layout(); plt.savefig('../assets/error_by_weather.png',dpi=150,bbox_inches='tight',facecolor='#0d1117'); plt.show()

## Worst geohash regions (top-10 by MAE)

In [ ]:
gh_err=train.groupby('gh4').agg(mae=('abs_err','mean'),mean_demand=('demand','mean'),count=('demand','size')).query('count>=20').sort_values('mae',ascending=False).head(10)
print(gh_err.round(4).to_string())

## Residual distribution

In [ ]:
fig,ax=plt.subplots(figsize=(10,4)); fig.set_facecolor('#0d1117')
train['residual'].hist(bins=100,ax=ax,color='#58a6ff',alpha=0.8,edgecolor='none')
ax.axvline(0,color='#f85149',ls='--',lw=1)
ax.set_xlabel('Residual (actual - predicted)',color='#c9d1d9')
ax.set_title('Residual Distribution',color='#e6edf3',fontweight='bold')
ax.set_facecolor('#0d1117'); ax.tick_params(colors='#8b949e')
for s in ax.spines.values(): s.set_color('#30363d')
plt.tight_layout(); plt.savefig('../assets/residual_dist.png',dpi=150,bbox_inches='tight',facecolor='#0d1117'); plt.show()

---
### Key findings
- Errors are highest during **peak hours** (the model slightly under-predicts demand spikes).
- Certain **road types** have systematically higher error — these carry less training data.
- The residual distribution is **near-zero-centered** (low bias) but has a right tail from under-predicted peaks.